# 02 · Modelado, calibración y política de aprobación — Andina Crédito

**Objetivo:** estimar la probabilidad de default de cada solicitud de `test.csv` y traducirla
a una política de aprobación con ganancia cuantificada.

Contenido:
1. Decisiones heredadas del EDA.
2. Esquema de validación temporal.
3. **Baseline interpretable**: scorecard logístico con WOE.
4. **LightGBM** con restricciones de monotonía.
5. **Explicabilidad**: importancia, SHAP, verificación de monotonía y explicación individual.
6. Calibración.
7. Política de aprobación y ganancia estimada.
8. Modelo final y `predictions.csv`.

> **Criterio transversal de este notebook: explicabilidad.** Un modelo de riesgo crediticio
> tiene que poder explicarse ante el gerente de Riesgo, ante auditoría y ante el cliente al que
> se le rechaza. Cada elección metodológica se justifica y cada resultado se acompaña de la
> evidencia que permite verificarlo.

In [ ]:
import sys, os, time, itertools, warnings
sys.path.append(os.path.join('..', 'src'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve

from prepare import preparar_datos

pio.renderers.default = 'notebook'

# ---- Paleta corporativa Banco BICE ----
BICE_AZUL       = '#0E162A'
BICE_AZUL_MED   = '#2E536D'
BICE_AZUL_CLARO = '#A9BDDF'
BICE_ACENTO     = '#CE894D'
BICE_PETROLEO   = '#062C33'
BICE_ALERTA     = '#C00000'
BICE_OK         = '#2E7D32'
ESCALA_BICE = ['#A9BDDF', '#2E536D', '#0E162A']

SEMILLA = 42
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

train_raw = pd.read_csv('../data/train.csv')
test_raw  = pd.read_csv('../data/test.csv')
train, test, params = preparar_datos(train_raw, test_raw)
train['fecha'] = pd.to_datetime(train['fecha_solicitud'])
test['fecha']  = pd.to_datetime(test['fecha_solicitud'])

print(f'train {train.shape}  |  test {test.shape}')
print(f'parámetros de limpieza: {params}')

## 1. Decisiones heredadas del EDA

Este notebook no vuelve a discutir lo ya resuelto en `01_eda.ipynb`. Las decisiones que
condicionan todo lo que sigue:

| Decisión | Origen | Consecuencia aquí |
|---|---|---|
| Excluir `num_contactos_ult_trimestre` | §6.2 — fuga de información | No entra al set de features. Cuesta ~0,14 de AUC declarado y es lo correcto. |
| Validación **out-of-time**, nunca k-fold aleatorio | §8, §8.3 | Folds temporales, también para elegir hiperparámetros. |
| Calibrar es más importante que el AUC | §9 | Isotónica obligatoria; se reporta Brier y curva de calibración. |
| Evaluar **con y sin** `tasa_interes_anual` | §7.4 — pricing derivado | Dos sets de features en competencia. |
| Umbral de aprobación por plazo | §9.2 | La política no usa un corte único. |

**Sin rebalanceo de clases.** Con 9,9% de positivos la tentación es SMOTE o `scale_pos_weight`.
Sería un error en este problema: cualquier rebalanceo distorsiona las probabilidades hacia
arriba, y toda la política depende de probabilidades **absolutas**. Los árboles manejan bien
este nivel de desbalance, y si igual se usaran pesos habría que deshacerlos al calibrar. Se
usa el desbalance natural.

In [ ]:
TARGET = 'default_12m'
CAT = ['tipo_empleo', 'region', 'canal', 'dia_semana_solicitud']
EXCLUIR = ['id_solicitud', 'fecha_solicitud', 'fecha', TARGET,
           'num_contactos_ult_trimestre']              # <- la fuga, excluida por decisión del EDA

FEATS_COMPLETO = [c for c in train.columns if c not in EXCLUIR]
FEATS_SIN_TASA = [c for c in FEATS_COMPLETO if c != 'tasa_interes_anual']
NUM = [c for c in FEATS_COMPLETO if c not in CAT]

print(f'Set completo   : {len(FEATS_COMPLETO)} variables')
print(f'Set sin la tasa: {len(FEATS_SIN_TASA)} variables')
print(f'\nCategóricas: {CAT}')


def X(df, feats):
    """Matriz de features con las categóricas tipadas para LightGBM."""
    M = df[feats].copy()
    for c in CAT:
        if c in M.columns:
            M[c] = M[c].astype('category')
    return M


def ks(y_true, y_score):
    """Estadístico KS: máxima separación entre las acumuladas de buenos y malos.
    Es la métrica estándar de la industria bancaria para scorecards."""
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(np.max(tpr - fpr))


def metricas(y, p, nombre=''):
    return {'modelo': nombre, 'AUC': roc_auc_score(y, p), 'KS': ks(y, p),
            'Brier': brier_score_loss(y, p)}

## 2. Esquema de validación temporal

Cuatro folds con ventana expansiva, exactamente los definidos en §8.3 del EDA: el
entrenamiento crece y la validación avanza en el tiempo, imitando el uso en producción.
**Los hiperparámetros también se eligen sobre estos folds**, no sobre folds aleatorios: si la
búsqueda se hiciera al azar, la elección ya quedaría contaminada aunque después se validara
temporalmente.

In [ ]:
FOLDS = [('2024-08-01', '2024-10-01'), ('2024-10-01', '2024-12-01'),
         ('2024-12-01', '2025-02-01'), ('2025-02-01', '2025-03-01')]

def particiones(df):
    """Genera (train_fold, valid_fold) para cada corte temporal."""
    for ini_va, fin_va in FOLDS:
        yield df[df.fecha < ini_va], df[(df.fecha >= ini_va) & (df.fecha < fin_va)]

for i, (tr_f, va_f) in enumerate(particiones(train), 1):
    print(f'fold {i}: entrena {len(tr_f):>6,} (→{tr_f.fecha.max().date()})  '
          f'valida {len(va_f):>5,} ({va_f.fecha.min().date()} → {va_f.fecha.max().date()})  '
          f'tasa validación {va_f[TARGET].mean():.2%}')

La tasa de default sube fold a fold: es la deriva documentada en §8. Cada fold es un problema
algo más difícil que el anterior, y el **fold 4 es el más representativo** del período de test.

## 3. Baseline interpretable: scorecard logístico con WOE

Antes de un modelo de árboles conviene tener un piso construido con la técnica estándar de la
banca: **regresión logística sobre variables transformadas a WOE** (*Weight of Evidence*).

**Qué es el WOE.** Cada variable se corta en tramos y cada tramo se reemplaza por

$$\text{WOE} = \ln\left(\frac{\%\text{ de buenos en el tramo}}{\%\text{ de malos en el tramo}}\right)$$

Valores positivos indican tramos con menos riesgo que el promedio; negativos, más riesgo.

**Por qué se usa en banca.** Tres razones prácticas: linealiza la relación con el logit, de
modo que la logística puede capturar relaciones no lineales sin perder interpretabilidad;
trata los faltantes como un tramo más, sin imputar; y produce un modelo que se puede expresar
como una **tabla de puntos** que un analista de riesgo lee sin saber estadística.

**El IV (*Information Value*)** resume el poder predictivo de cada variable: bajo 0,02 es
inútil, sobre 0,3 es fuerte, y sobre 0,5 conviene sospechar de fuga.

In [ ]:
def calcular_woe(df, var, y, n_bins=8, es_cat=False):
    """Tabla WOE/IV de una variable, calculada SOLO sobre el fold de entrenamiento."""
    if es_cat:
        tramo = df[var].astype(str)
        cortes = None
    else:
        cortes = np.unique(np.nanquantile(df[var], np.linspace(0, 1, n_bins + 1)))
        cortes[0], cortes[-1] = -np.inf, np.inf
        tramo = pd.cut(df[var], cortes)
    g = pd.DataFrame({'tramo': tramo, 'y': y}).groupby('tramo', observed=True)['y'].agg(['size', 'sum'])
    g.columns = ['n', 'malos']
    g['buenos'] = g['n'] - g['malos']
    # corrección de Laplace para evitar log(0) en tramos sin eventos
    g['p_buenos'] = (g['buenos'] + 0.5) / (g['buenos'].sum() + 0.5 * len(g))
    g['p_malos']  = (g['malos']  + 0.5) / (g['malos'].sum()  + 0.5 * len(g))
    g['woe'] = np.log(g['p_buenos'] / g['p_malos'])
    g['iv_parcial'] = (g['p_buenos'] - g['p_malos']) * g['woe']
    return g.reset_index(), cortes, float(g['iv_parcial'].sum())


def ajustar_woe(df_tr, y_tr, feats):
    """Ajusta el binning WOE sobre el fold de entrenamiento. Devuelve el 'mapa' y los IV."""
    mapa, ivs = {}, {}
    for v in feats:
        es_cat = v in CAT
        tabla, cortes, iv = calcular_woe(df_tr, v, y_tr, es_cat=es_cat)
        mapa[v] = {'tabla': tabla, 'cortes': cortes, 'es_cat': es_cat}
        ivs[v] = iv
    return mapa, pd.Series(ivs).sort_values(ascending=False)


def aplicar_woe(df, mapa):
    """Transforma un DataFrame usando el mapa ajustado en train (sin leakage)."""
    out = pd.DataFrame(index=df.index)
    for v, m in mapa.items():
        tabla = m['tabla'].set_index('tramo')['woe']
        if m['es_cat']:
            out[v] = df[v].astype(str).map(tabla).astype(float)
        else:
            tramo = pd.cut(df[v], m['cortes'])
            out[v] = tramo.map(tabla).astype(float)
    return out.fillna(0.0)   # categoría no vista en train -> WOE neutro


tr_f, va_f = list(particiones(train))[-1]      # fold más reciente, el más representativo
mapa_woe, iv = ajustar_woe(tr_f, tr_f[TARGET], FEATS_COMPLETO)

fig = go.Figure(go.Bar(
    x=iv.values[::-1], y=iv.index[::-1], orientation='h',
    marker_color=[BICE_ACENTO if v > 0.3 else BICE_AZUL_MED if v > 0.1 else BICE_AZUL_CLARO
                  for v in iv.values[::-1]],
    text=[f'{v:.3f}' for v in iv.values[::-1]], textposition='outside'))
for x_, txt in [(0.02, 'inútil'), (0.1, 'débil'), (0.3, 'fuerte')]:
    fig.add_vline(x=x_, line_dash='dot', line_color=BICE_PETROLEO,
                  annotation_text=txt, annotation_position='top')
fig.update_layout(title='Information Value por variable (calculado en el fold de entrenamiento)',
                  xaxis_title='IV', yaxis_title='', height=640, margin=dict(l=250), showlegend=False)
fig.show()

print(iv.round(4).to_string())

**Lectura:** `score_buro` domina, seguido de `uso_linea_credito_pct`, `tasa_interes_anual` y
`num_consultas_buro_3m`. Ningún IV supera 0,5, lo que confirma que **no queda ninguna otra
fuga** en el set después de excluir `num_contactos_ult_trimestre`. Las banderas de limpieza
aportan poco, como se esperaba: son correcciones de calidad, no señal de riesgo.

### 3.1 La tabla WOE de la variable más importante

Así es como un analista de riesgo lee el modelo: tramo por tramo, con el volumen y la tasa de
mora de cada uno. Esta es la ventaja del scorecard sobre cualquier modelo de caja negra.

In [ ]:
tabla_score = mapa_woe['score_buro']['tabla'].copy()
tabla_score['tasa_default'] = tabla_score['malos'] / tabla_score['n']
tabla_score['tramo_txt'] = tabla_score['tramo'].astype(str)

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_bar(x=tabla_score['tramo_txt'], y=tabla_score['n'], name='volumen',
            marker_color=BICE_AZUL_CLARO, opacity=0.6, secondary_y=True)
fig.add_scatter(x=tabla_score['tramo_txt'], y=tabla_score['woe'], mode='lines+markers+text',
                name='WOE', line=dict(color=BICE_ACENTO, width=3), marker=dict(size=10),
                text=tabla_score['woe'].round(2), textposition='top center', secondary_y=False)
fig.add_hline(y=0, line_dash='dot', line_color=BICE_PETROLEO, secondary_y=False)
fig.update_yaxes(title_text='WOE (positivo = menos riesgo)', secondary_y=False)
fig.update_yaxes(title_text='n° de solicitudes', secondary_y=True)
fig.update_layout(title=f"WOE por tramo de score_buro · IV = {iv['score_buro']:.3f}",
                  xaxis_title='tramo de score', xaxis_tickangle=-30, height=430)
fig.show()

print(tabla_score[['tramo_txt', 'n', 'tasa_default', 'woe']].to_string(index=False))

**Hallazgo:** la relación es **perfectamente monótona** — a mayor score, mayor WOE, menor
riesgo. Esto es exactamente lo que se espera del negocio y es la evidencia empírica que
justifica imponer restricciones de monotonía al modelo de árboles en la sección 4.

### 3.2 Ajuste del baseline y desempeño por fold

In [ ]:
def evaluar_baseline(feats, nombre):
    filas = []
    for i, (tr_f, va_f) in enumerate(particiones(train), 1):
        mapa, _ = ajustar_woe(tr_f, tr_f[TARGET], feats)          # binning solo con train del fold
        Xtr, Xva = aplicar_woe(tr_f, mapa), aplicar_woe(va_f, mapa)
        lr = LogisticRegression(max_iter=1000, C=1.0).fit(Xtr, tr_f[TARGET])
        p = lr.predict_proba(Xva)[:, 1]
        filas.append({'fold': i, **metricas(va_f[TARGET], p, nombre)})
    return pd.DataFrame(filas)

res_base = evaluar_baseline(FEATS_COMPLETO, 'Logística WOE')
print(res_base.round(4).to_string(index=False))
print(f"\nAUC promedio: {res_base.AUC.mean():.4f} ± {res_base.AUC.std():.4f}"
      f"   |   fold 4 (el más representativo): {res_base.AUC.iloc[-1]:.4f}")

## 4. LightGBM

### 4.1 Por qué LightGBM y no otro algoritmo

La elección de familia de modelo vale poco en este problema —del orden de 0,01 de AUC— y
conviene decirlo con esas palabras. Aun así, la decisión se justifica:

| Alternativa | Por qué se descarta como modelo principal |
|---|---|
| **Regresión logística** | Va como baseline, no como modelo final. Solo captura relaciones no lineales si se codifican a mano (vía WOE) y solo ve interacciones si se escriben explícitamente. En riesgo crediticio esas interacciones importan: score alto con uso de línea al tope no es lo mismo que score alto con uso bajo. |
| **Random Forest** | Dos problemas concretos aquí: sus probabilidades salen de promediar votos, lo que las comprime hacia el centro y las deja mal calibradas de origen — justo lo que más importa en este problema. Y **no soporta restricciones de monotonía**, que es una decisión de diseño de este proyecto. |
| **Redes neuronales** | En datos tabulares con 45.000 filas los árboles con boosting siguen ganando de forma consistente. Sumado a que no admiten monotonía de forma limpia ni se explican ante el gerente de Riesgo, no hay argumento a favor. |
| **XGBoost** | Rendiría casi igual. Se prefiere LightGBM por velocidad: la búsqueda son 4 folds × 2 sets de features × 24 configuraciones ≈ 200 ajustes, y el crecimiento *leaf-wise* con histogramas es notoriamente más rápido. Su soporte nativo de categóricas también está más maduro. |
| **CatBoost** | **Sería una elección igual de defendible**, probablemente mejor sin tuning. Su ventaja está en categóricas de alta cardinalidad, y aquí la máxima es `region` con 16 niveles: con esa cardinalidad la diferencia se diluye. Empate técnico resuelto por presupuesto de tiempo. |

**Lo que sí mueve la aguja** en este problema son tres decisiones ya tomadas en el EDA:
excluir la fuga (0,14 de AUC entre lo declarado y lo real), validar temporalmente y calibrar.
Comparar cinco algoritmos y no detectar `num_contactos_ult_trimestre` produce un trabajo peor
que usar el primer LightGBM razonable y sí verlo. Probar CatBoost y XGBoost queda como
próximo paso explícito.

### 4.2 Restricciones de monotonía

LightGBM permite forzar que la probabilidad predicha se mueva **siempre** en una dirección
respecto de una variable. Se imponen donde el negocio tiene una expectativa inequívoca:

| Variable | Restricción | Razón de negocio |
|---|---|---|
| `score_buro` | ↓ riesgo | Mejor score no puede implicar más riesgo. |
| `peor_morosidad_12m` | ↑ riesgo | Más días de mora previa no puede implicar menos riesgo. |
| `uso_linea_credito_pct` | ↑ riesgo | Mayor utilización de la línea es señal de estrés financiero. |
| `num_consultas_buro_3m` | ↑ riesgo | Más consultas recientes indican búsqueda activa de crédito. |
| `num_creditos_vigentes` | ↑ riesgo | Mayor exposición simultánea. |
| `ratio_deuda_ingreso` | ↑ riesgo | Menor capacidad de pago. |
| `antiguedad_cliente_meses` | ↓ riesgo | Más historia con la institución reduce incertidumbre. |

**Por qué vale la pena aunque cueste métrica.** Tres razones: (a) **robustez bajo deriva** — el
modelo no puede aprender un pliegue espurio en una región donde la población está cambiando,
que es exactamente el riesgo documentado en §8; (b) **defendibilidad** — nunca habrá que
explicarle al gerente de Riesgo por qué un cliente con mejor score recibió peor probabilidad;
(c) **cumplimiento** — un modelo monótono se audita y se documenta con mucha menos fricción.

Al resto de las variables (edad, monto, plazo, ingreso, categóricas) **no** se les impone nada:
su relación con el riesgo no es monótona a priori y forzarla sería un supuesto injustificado.

In [ ]:
MONOTONIA = {
    'score_buro': -1, 'antiguedad_cliente_meses': -1,
    'peor_morosidad_12m': 1, 'uso_linea_credito_pct': 1, 'num_consultas_buro_3m': 1,
    'num_creditos_vigentes': 1, 'ratio_deuda_ingreso': 1,
}

def restricciones(feats):
    return [MONOTONIA.get(c, 0) for c in feats]

print('Restricciones aplicadas:')
for c, v in MONOTONIA.items():
    print(f'  {c:<28} {"↓ riesgo" if v < 0 else "↑ riesgo"}')
print(f'\nSin restricción (libres): '
      f'{[c for c in FEATS_COMPLETO if c not in MONOTONIA]}')

### 4.3 Búsqueda de hiperparámetros sobre los folds temporales

In [ ]:
GRILLA = {
    'num_leaves':        [15, 31, 63],
    'min_child_samples': [30, 60, 120],
    'learning_rate':     [0.03, 0.05],
    'reg_lambda':        [0.0, 5.0],
    'colsample_bytree':  [0.7, 1.0],
}
rng = np.random.default_rng(SEMILLA)
combos = list(itertools.product(*GRILLA.values()))
muestra = [dict(zip(GRILLA, combos[i])) for i in rng.choice(len(combos), size=24, replace=False)]
print(f'{len(combos)} combinaciones posibles → se evalúan {len(muestra)} al azar sobre 4 folds temporales')


def cv_temporal(feats, hp, con_monotonia=True, semilla=SEMILLA):
    """AUC por fold. El binning/ajuste ocurre solo con el train de cada fold."""
    aucs = []
    for tr_f, va_f in particiones(train):
        m = lgb.LGBMClassifier(
            n_estimators=500, verbose=-1, random_state=semilla,
            monotone_constraints=restricciones(feats) if con_monotonia else None, **hp)
        m.fit(X(tr_f, feats), tr_f[TARGET],
              eval_set=[(X(va_f, feats), va_f[TARGET])], eval_metric='auc',
              callbacks=[lgb.early_stopping(50, verbose=False)])
        aucs.append(roc_auc_score(va_f[TARGET], m.predict_proba(X(va_f, feats))[:, 1]))
    return np.array(aucs)


t0 = time.time()
busqueda = []
for hp in muestra:
    a = cv_temporal(FEATS_COMPLETO, hp)
    busqueda.append({**hp, 'auc_medio': a.mean(), 'auc_std': a.std(), 'auc_fold4': a[-1]})
busqueda = pd.DataFrame(busqueda).sort_values('auc_medio', ascending=False).reset_index(drop=True)
print(f'búsqueda completada en {time.time()-t0:.0f}s\n')
print(busqueda.head(8).round(4).to_string(index=False))

MEJOR = {k: busqueda.loc[0, k] for k in GRILLA}
MEJOR['num_leaves'] = int(MEJOR['num_leaves']); MEJOR['min_child_samples'] = int(MEJOR['min_child_samples'])
print(f'\nMejor configuración: {MEJOR}')

In [ ]:
fig = go.Figure()
fig.add_scatter(x=busqueda.index, y=busqueda.auc_medio, mode='markers',
                error_y=dict(type='data', array=busqueda.auc_std, color=BICE_AZUL_CLARO),
                marker=dict(color=BICE_AZUL_MED, size=9), name='AUC medio ± desv. entre folds')
fig.add_scatter(x=busqueda.index, y=busqueda.auc_fold4, mode='markers',
                marker=dict(color=BICE_ACENTO, size=7, symbol='diamond'), name='AUC fold 4')
fig.add_vline(x=0, line_dash='dot', line_color=BICE_ALERTA, annotation_text='elegida')
fig.update_layout(title='Búsqueda de hiperparámetros — las 24 configuraciones ordenadas por AUC medio',
                  xaxis_title='configuración (ordenada)', yaxis_title='AUC out-of-time', height=430)
fig.show()

**Lectura:** la diferencia entre la mejor y la peor configuración es pequeña frente a la
dispersión **entre folds** de una misma configuración. Traducido: el modelo es poco sensible
al tuning y muy sensible al período en que se lo evalúa. Eso refuerza que la incertidumbre
relevante aquí es temporal, no de hiperparámetros — y justifica no gastar más presupuesto en
búsqueda.

### 4.4 ¿Cuánto cuesta la monotonía? ¿Y excluir la tasa?

Dos preguntas que el EDA dejó abiertas, resueltas con el mismo protocolo de validación.

In [ ]:
comparacion = {}
for nombre, feats, mono in [
        ('LightGBM completo (sin monotonía)', FEATS_COMPLETO, False),
        ('LightGBM completo (con monotonía)', FEATS_COMPLETO, True),
        ('LightGBM sin tasa (con monotonía)', FEATS_SIN_TASA,  True)]:
    comparacion[nombre] = cv_temporal(feats, MEJOR, con_monotonia=mono)

comp = pd.DataFrame({k: v for k, v in comparacion.items()},
                    index=[f'fold {i}' for i in range(1, 5)]).T
comp['promedio'] = comp.mean(axis=1)
print(comp.round(4).to_string())

fig = go.Figure()
for i, (nombre, aucs) in enumerate(comparacion.items()):
    fig.add_scatter(x=[f'fold {j}' for j in range(1, 5)], y=aucs, mode='lines+markers',
                    name=nombre, line=dict(width=2.5,
                    color=[BICE_AZUL_CLARO, BICE_AZUL, BICE_ACENTO][i]), marker=dict(size=9))
fig.add_scatter(x=[f'fold {j}' for j in range(1, 5)], y=res_base.AUC.values, mode='lines+markers',
                name='Baseline logística WOE', line=dict(color=BICE_PETROLEO, dash='dot', width=2),
                marker=dict(size=8, symbol='square'))
fig.update_layout(title='AUC out-of-time por fold — todas las variantes',
                  xaxis_title='', yaxis_title='AUC', height=450)
fig.show()

**Hallazgos:**

- **La monotonía no cuesta métrica; en estos datos incluso ayuda.** Es el mejor de los mundos:
  un modelo más defendible y más robusto sin sacrificar desempeño. La explicación es que las
  restricciones actúan como regularización, impidiendo pliegues que el modelo aprendería del
  ruido.
- **Excluir `tasa_interes_anual` cuesta muy poco**, tal como anticipaba el §7.4 del EDA, donde
  el AUC del residuo de la tasa tras remover el score era 0,533. Se confirma la hipótesis: la
  tasa es señal derivada.
- **El baseline logístico queda cerca.** La ganancia de LightGBM es real pero moderada. Se
  reporta así en el informe: un scorecard tradicional resolvería el 95% del problema.

**Decisión sobre la tasa:** se usa el set **sin `tasa_interes_anual`**. Cuesta una fracción de
AUC y compra independencia del motor de pricing: si mañana Andina cambia su política de
precios, el modelo no se degrada. Es una decisión de robustez, no de métrica.

In [ ]:
FEATS = FEATS_SIN_TASA
print(f'Set de features definitivo: {len(FEATS)} variables (sin tasa_interes_anual, sin la fuga)')

## 5. Explicabilidad

Un modelo de riesgo tiene que explicarse en tres niveles distintos, y cada uno necesita una
herramienta diferente:

| Nivel | Pregunta | Herramienta |
|---|---|---|
| **Global** | ¿Qué variables usa el modelo? | Importancia por ganancia + SHAP |
| **Estructural** | ¿Se comporta como el negocio espera? | Curvas de efecto + verificación de monotonía |
| **Individual** | ¿Por qué se rechazó a *esta* persona? | SHAP por caso |

El tercero no es opcional: si la política rechaza solicitudes, alguien va a preguntar por qué.

In [ ]:
tr_f, va_f = list(particiones(train))[-1]
modelo = lgb.LGBMClassifier(n_estimators=500, verbose=-1, random_state=SEMILLA,
                            monotone_constraints=restricciones(FEATS), **MEJOR)
modelo.fit(X(tr_f, FEATS), tr_f[TARGET],
           eval_set=[(X(va_f, FEATS), va_f[TARGET])], eval_metric='auc',
           callbacks=[lgb.early_stopping(50, verbose=False)])
print(f'árboles usados: {modelo.best_iteration_}  |  '
      f'AUC fold 4: {roc_auc_score(va_f[TARGET], modelo.predict_proba(X(va_f, FEATS))[:,1]):.4f}')

### 5.1 Importancia global

In [ ]:
imp = pd.Series(modelo.booster_.feature_importance('gain'), index=FEATS).sort_values()
imp_pct = imp / imp.sum()

fig = go.Figure(go.Bar(x=imp_pct.values, y=imp_pct.index, orientation='h',
                       marker_color=BICE_AZUL_MED,
                       text=[f'{v:.1%}' for v in imp_pct.values], textposition='outside'))
fig.update_layout(title='Importancia por ganancia — cuánto reduce el error cada variable',
                  xaxis_title='% de la ganancia total', xaxis_tickformat='.0%',
                  yaxis_title='', height=640, margin=dict(l=250))
fig.show()
print(imp_pct.sort_values(ascending=False).head(8).apply(lambda v: f'{v:.2%}').to_string())

### 5.2 SHAP: dirección y magnitud del efecto

La importancia por ganancia dice *cuánto* pesa cada variable, pero no *en qué dirección* ni
para quién. SHAP reparte la predicción de cada caso entre las variables, de forma que las
contribuciones suman exactamente la predicción. Permite dos lecturas: el efecto promedio
(global) y la explicación de un caso puntual (individual, §5.4).

In [ ]:
import shap

muestra_shap = va_f.sample(min(2500, len(va_f)), random_state=SEMILLA)
explainer = shap.TreeExplainer(modelo)
sv = explainer.shap_values(X(muestra_shap, FEATS))
if isinstance(sv, list):
    sv = sv[1]

orden = pd.Series(np.abs(sv).mean(axis=0), index=FEATS).sort_values(ascending=False).head(10).index

fig = go.Figure()
for i, v in enumerate(orden[::-1]):
    j = FEATS.index(v)
    val = muestra_shap[v]
    color = (pd.factorize(val)[0] if v in CAT else pd.to_numeric(val, errors='coerce'))
    color = pd.Series(color).rank(pct=True)
    fig.add_scatter(x=sv[:, j], y=np.full(len(sv), i) + np.random.uniform(-0.18, 0.18, len(sv)),
                    mode='markers', name=v, showlegend=False,
                    marker=dict(size=4, opacity=0.55, color=color, colorscale=ESCALA_BICE,
                                showscale=(i == len(orden)-1),
                                colorbar=dict(title='valor de<br>la variable', tickvals=[0, 1],
                                              ticktext=['bajo', 'alto'])),
                    hovertext=[f'{v}={x}' for x in val], hoverinfo='text+x')
fig.add_vline(x=0, line_color=BICE_PETROLEO, line_dash='dot')
fig.update_layout(title='Efecto SHAP por variable — a la derecha aumenta el riesgo predicho',
                  xaxis_title='contribución al log-odds de default',
                  yaxis=dict(tickmode='array', tickvals=list(range(len(orden))),
                             ticktext=list(orden[::-1])),
                  height=560, margin=dict(l=250))
fig.show()

**Cómo leerlo:** cada punto es una solicitud. A la derecha del cero, esa variable **empujó al
alza** el riesgo de ese cliente. El color indica si el valor de la variable era alto u bajo.
Un patrón limpio —oscuros a un lado, claros al otro— confirma que el efecto es direccional y
consistente, no ruido.

### 5.3 Verificación: ¿se cumplen las restricciones de monotonía?

Imponer una restricción no basta: hay que comprobar que el modelo entregado la respeta.

**Cuál es la herramienta correcta — y cuál no.** El primer intento fue verificarlo con el
efecto SHAP promedio por decil, y **dio falsos incumplimientos** en tres variables. El error
era del método, no del modelo: el SHAP de una variable absorbe interacciones con las demás, y
al agrupar por deciles se mezclan casos con perfiles distintos, así que el promedio puede
subir y bajar aunque cada predicción individual sea perfectamente monótona.

Lo que la restricción de LightGBM garantiza es otra cosa: que al mover **solo** esa variable,
manteniendo el resto fijo, la predicción se mueva siempre en la misma dirección. La
herramienta que mide exactamente eso es la **dependencia parcial**: se recorre una grilla de
valores de la variable, se aplica a una muestra real de solicitudes dejando todo lo demás
intacto, y se promedia la predicción.

In [ ]:
base_pdp = X(va_f.sample(min(400, len(va_f)), random_state=SEMILLA), FEATS)
restringidas = [v for v in MONOTONIA if v in FEATS]

fig = make_subplots(rows=2, cols=4, subplot_titles=[
    f'{v} ({"↓" if MONOTONIA[v] < 0 else "↑"})' for v in restringidas])
resultados_mono = {}
for k, v in enumerate(restringidas):
    grid = np.unique(np.quantile(train[v], np.linspace(0.01, 0.99, 20)))
    pdp = []
    for g in grid:
        tmp = base_pdp.copy(); tmp[v] = g
        pdp.append(modelo.predict_proba(tmp)[:, 1].mean())
    pdp = np.array(pdp); dif = np.diff(pdp)
    ok = bool((dif <= 1e-12).all()) if MONOTONIA[v] < 0 else bool((dif >= -1e-12).all())
    resultados_mono[v] = ok
    r, c = k // 4 + 1, k % 4 + 1
    fig.add_scatter(x=grid, y=pdp, mode='lines+markers', showlegend=False,
                    line=dict(color=BICE_OK if ok else BICE_ALERTA, width=2.5),
                    marker=dict(size=6), row=r, col=c)
fig.update_yaxes(tickformat='.1%')
fig.update_layout(
    title='Verificación de monotonía por dependencia parcial<br>'
          '<sup>Cada curva mueve solo esa variable y deja el resto fijo · verde = restricción respetada</sup>',
    height=620)
fig.update_annotations(font_size=11)
fig.show()

for v, ok in resultados_mono.items():
    print(f'  {v:<28} {"↓" if MONOTONIA[v] < 0 else "↑"}  monotonía respetada: {ok}')
assert all(resultados_mono.values()), 'alguna restricción de monotonía no se está respetando'
print('\nLas 7 restricciones se cumplen sin excepción.')

**Hallazgo:** las siete restricciones se respetan. Además las curvas son informativas por sí
mismas: `score_buro` mueve la probabilidad de ~27% a ~1,6% de un extremo al otro, mientras que
`num_creditos_vigentes` apenas la mueve un punto. Eso dice qué variables realmente gobiernan la
decisión, y coincide con la importancia por ganancia de §5.1.

**La lección metodológica vale para el informe:** SHAP explica *predicciones*, la dependencia
parcial describe el *comportamiento del modelo*. Son preguntas distintas y usar la herramienta
equivocada produce conclusiones equivocadas — en este caso, tres alertas falsas.

### 5.4 Explicabilidad individual: por qué se rechazó a este cliente

El caso de uso operativo: una solicitud recibe una probabilidad alta y alguien pregunta por
qué. SHAP descompone esa predicción en la contribución de cada variable, y las
contribuciones suman exactamente el resultado.

In [ ]:
p_va = modelo.predict_proba(X(muestra_shap, FEATS))[:, 1]
idx = int(np.argsort(p_va)[-8])     # un caso claramente riesgoso, no el extremo
caso = muestra_shap.iloc[idx]
contrib = pd.Series(sv[idx], index=FEATS).sort_values(key=np.abs, ascending=False).head(9)[::-1]
etiquetas = [f'{v} = {caso[v]:,.0f}' if v not in CAT else f'{v} = {caso[v]}' for v in contrib.index]

fig = go.Figure(go.Bar(x=contrib.values, y=etiquetas, orientation='h',
                       marker_color=[BICE_ALERTA if v > 0 else BICE_OK for v in contrib.values],
                       text=[f'{v:+.2f}' for v in contrib.values], textposition='outside'))
fig.add_vline(x=0, line_color=BICE_PETROLEO)
fig.update_layout(
    title=f'Explicación individual · solicitud {caso.id_solicitud} — '
          f'probabilidad predicha {p_va[idx]:.1%}<br>'
          '<sup>Rojo = empuja al rechazo · Verde = empuja a la aprobación</sup>',
    xaxis_title='contribución al log-odds', yaxis_title='', height=470, margin=dict(l=290))
fig.show()

umbral_caso = (0.005*caso.plazo_meses)/(0.005*caso.plazo_meses + 0.55)
print(f'Solicitud {caso.id_solicitud}: plazo {caso.plazo_meses} meses → umbral {umbral_caso:.1%}')
print(f'Probabilidad predicha {p_va[idx]:.1%} → decisión: '
      f'{"RECHAZAR" if p_va[idx] > umbral_caso else "APROBAR"}')
print(f'Resultado real observado: {"cayó en default" if caso[TARGET]==1 else "pagó"}')

**Esto es lo que se le entrega al analista de riesgo:** no "el modelo dijo 62%", sino "el score
de buró y la utilización de línea de este cliente lo empujan al rechazo, mientras que su
antigüedad como cliente lo compensa parcialmente". Es auditable y comunicable.

## 6. Calibración

### 6.1 El conflicto de diseño, y cómo se resuelve

Para predecir sobre test conviene entrenar con **toda** la data hasta 2025-02: los meses
recientes son los más parecidos al período a scorear. Pero la isotónica necesita un bloque que
el modelo **no** haya visto; si se ajusta sobre datos de entrenamiento, sale optimista y no
corrige nada.

**Solución adoptada:** predicciones *out-of-fold* generadas temporalmente. Cada bloque de
validación se predice con un modelo que no lo vio, se concatenan esas predicciones y la
isotónica se ajusta sobre ellas. Así se usa toda la data disponible y ninguna predicción de
calibración proviene de datos vistos.

**Limitación honesta:** esas predicciones mezclan regímenes de riesgo distintos (2024-08 tiene
10,6% de mora, 2025-02 tiene 12,7%), así que la isotónica aprende un promedio del período. Se
documenta como tal.

In [ ]:
oof = []
for tr_f, va_f in particiones(train):
    m = lgb.LGBMClassifier(n_estimators=500, verbose=-1, random_state=SEMILLA,
                           monotone_constraints=restricciones(FEATS), **MEJOR)
    m.fit(X(tr_f, FEATS), tr_f[TARGET], eval_set=[(X(va_f, FEATS), va_f[TARGET])],
          eval_metric='auc', callbacks=[lgb.early_stopping(50, verbose=False)])
    oof.append(pd.DataFrame({'p': m.predict_proba(X(va_f, FEATS))[:, 1],
                             'y': va_f[TARGET].values, 'fecha': va_f.fecha.values,
                             'monto_solicitado': va_f.monto_solicitado.values,
                             'plazo_meses': va_f.plazo_meses.values}))
oof = pd.concat(oof, ignore_index=True)
print(f'predicciones out-of-fold: {len(oof):,} desde {pd.to_datetime(oof.fecha).min().date()} '
      f'hasta {pd.to_datetime(oof.fecha).max().date()}')

calibrador = IsotonicRegression(out_of_bounds='clip').fit(oof.p, oof.y)
oof['p_cal'] = calibrador.predict(oof.p)

print(f"\n                AUC      KS      Brier    prob. media")
for nom, col in [('sin calibrar', 'p'), ('con isotónica', 'p_cal')]:
    m_ = metricas(oof.y, oof[col])
    print(f"{nom:<15} {m_['AUC']:.4f}  {m_['KS']:.4f}  {m_['Brier']:.5f}   {oof[col].mean():.2%}")
print(f"{'tasa real':<15} {'—':>6}  {'—':>6}  {'—':>7}   {oof.y.mean():.2%}")

In [ ]:
def curva_calib(p, y, n=10):
    b = pd.qcut(pd.Series(p), n, duplicates='drop')
    g = pd.DataFrame({'p': p.values, 'y': y.values}).groupby(b.values, observed=True)
    return g['p'].mean().values, g['y'].mean().values

x0, y0 = curva_calib(oof.p, oof.y)
x1, y1 = curva_calib(oof.p_cal, oof.y)
lim = max(x0.max(), y0.max(), x1.max())

fig = go.Figure()
fig.add_scatter(x=[0, lim], y=[0, lim], mode='lines', name='calibración perfecta',
                line=dict(color=BICE_PETROLEO, dash='dash'))
fig.add_scatter(x=x0, y=y0, mode='lines+markers', name='sin calibrar',
                line=dict(color=BICE_ALERTA, width=2.5), marker=dict(size=9))
fig.add_scatter(x=x1, y=y1, mode='lines+markers', name='con isotónica',
                line=dict(color=BICE_OK, width=2.5), marker=dict(size=9))
fig.update_layout(title='Curva de calibración sobre predicciones out-of-fold (deciles de riesgo)',
                  xaxis_title='probabilidad predicha', yaxis_title='tasa de default observada',
                  xaxis_tickformat='.0%', yaxis_tickformat='.0%', height=470)
fig.show()

### 6.2 Lo que NO se ajusta, y por qué se declara

La tasa de default sube ~0,46 puntos por mes (§8 del EDA). El último bloque de train está en
12,7% y la extrapolación para el período de test da **~14,4%**. Aunque la calibración sea
perfecta contra los datos disponibles, sigue calibrando contra un mundo que ya quedó atrás:
**el modelo va a subestimar el riesgo en test, por construcción**.

Se podría corregir desplazando el intercepto hacia la tasa proyectada. **Decisión: no hacerlo.**

Ese ajuste sería una apuesta a que la tendencia continúa linealmente durante cinco meses más,
y no hay forma de validarla con los datos disponibles. Si la tendencia se aplana, el ajuste
rechaza clientes rentables. Aplicar una corrección no verificable y reportarla como si fuera
parte del modelo sería precisamente el tipo de brecha entre lo declarado y lo real que hay que
evitar.

**En su lugar se declara explícitamente en el informe ejecutivo:**

> Las probabilidades entregadas probablemente subestiman el riesgo del período de test en
> torno a 1-2 puntos porcentuales, debido a la tendencia creciente de mora documentada en el
> análisis. La política de aprobación es, en ese margen, **menos conservadora** de lo que
> debería. Se recomienda recalibrar con los primeros resultados observados del período.

In [ ]:
tasa_ultimo = train[train.fecha >= '2025-01-01'][TARGET].mean()
mens = train.groupby(train.fecha.dt.to_period('M'))[TARGET].mean()
z = np.polyfit(np.arange(len(mens)), mens.values, 1)
proy = np.polyval(z, np.arange(len(mens), len(mens) + 5)).mean()

fig = go.Figure(go.Bar(
    x=['Media histórica<br>de train', 'Predicción media<br>del modelo (calibrada)',
       'Último bloque<br>observado (2025-01/02)', 'Proyección para<br>el período de test'],
    y=[train[TARGET].mean(), oof.p_cal.mean(), tasa_ultimo, proy],
    marker_color=[BICE_AZUL_CLARO, BICE_OK, BICE_AZUL_MED, BICE_ACENTO],
    text=[f'{v:.1%}' for v in [train[TARGET].mean(), oof.p_cal.mean(), tasa_ultimo, proy]],
    textposition='outside'))
fig.add_annotation(x=2.5, y=proy*1.06, showarrow=False, font=dict(color=BICE_ALERTA, size=12),
                   text=f'<b>brecha declarada: ~{(proy - oof.p_cal.mean())*100:.1f} pp</b>')
fig.update_layout(title='La brecha que se declara en vez de ajustar',
                  yaxis_title='tasa de default', yaxis_tickformat='.0%',
                  height=440, showlegend=False, yaxis_range=[0, proy*1.25])
fig.show()
print(f'Brecha estimada entre la predicción media y la proyección: '
      f'{(proy - oof.p_cal.mean())*100:.1f} puntos porcentuales')

## 7. Política de aprobación y ganancia estimada

Se aplica la regla derivada en §9 del EDA: **aprobar si `p` está bajo el umbral que
corresponde al plazo solicitado**, con `p* = 0,005·plazo / (0,005·plazo + 0,55)`.

In [ ]:
MM = 1e6
d = oof.copy()
d['G'] = d.monto_solicitado * 0.005 * d.plazo_meses
d['L'] = d.monto_solicitado * 0.55
d['margen'] = np.where(d.y == 1, -d.L, d.G)
d['umbral'] = d.G / (d.G + d.L)

politicas = {
    'Aprobar todo\n(situación base)': (d.margen.sum()/MM, 1.0),
    'Umbral por plazo\n(recomendada)': (d.loc[d.p_cal < d.umbral, 'margen'].sum()/MM,
                                        (d.p_cal < d.umbral).mean()),
    'Oráculo\n(si supiéramos el futuro)': (d.loc[d.y == 0, 'margen'].sum()/MM, (d.y == 0).mean()),
}
vals = [v[0] for v in politicas.values()]; aps = [v[1] for v in politicas.values()]

fig = go.Figure(go.Bar(x=list(politicas), y=vals,
                       marker_color=[BICE_AZUL_CLARO, BICE_ACENTO, BICE_PETROLEO],
                       text=[f'{v:,.0f} MM<br><span style="font-size:11px">aprueba {a:.0%}</span>'
                             for v, a in zip(vals, aps)], textposition='outside'))
fig.add_annotation(x=0.5, y=max(vals)*0.6, showarrow=False, font=dict(color=BICE_ALERTA, size=13),
                   text=f'<b>+{vals[1]-vals[0]:,.0f} MM</b><br>({(vals[1]/vals[0]-1)*100:.0f}%)')
fig.update_layout(title='Ganancia estimada de la política · backtest out-of-fold 2024-08 → 2025-02',
                  yaxis_title='ganancia (millones de CLP)', height=460, showlegend=False,
                  yaxis_range=[0, max(vals)*1.25])
fig.show()

for n, (v, a) in politicas.items():
    print(f'{n.splitlines()[0]:<22}: {v:>9,.0f} MM CLP  |  aprueba {a:.1%}')
print(f'\nGanancia incremental: {vals[1]-vals[0]:,.0f} MM CLP '
      f'({(vals[1]/vals[0]-1)*100:.0f}% sobre aprobar todo)')

**Nota de comparabilidad:** las cifras absolutas difieren de las de §9 del EDA porque aquí el
backtest cubre siete meses (2024-08 → 2025-02) en vez de cuatro, y usa el modelo definitivo en
lugar del diagnóstico. Lo que importa es que la conclusión se sostiene con otro modelo, otro
período y más datos: la política más que duplica la ganancia frente a aprobar todo.

## 8. Modelo final y generación de `predictions.csv`

El modelo final se entrena sobre **toda** la data de train (incluidos los meses más recientes,
los más parecidos al período de test), con la configuración elegida, las restricciones de
monotonía y el número de árboles determinado por el promedio de los folds. Las predicciones se
pasan por el calibrador isotónico ajustado en §6.

In [ ]:
n_arboles = int(np.mean([
    lgb.LGBMClassifier(n_estimators=500, verbose=-1, random_state=SEMILLA,
                       monotone_constraints=restricciones(FEATS), **MEJOR)
    .fit(X(tr_f, FEATS), tr_f[TARGET], eval_set=[(X(va_f, FEATS), va_f[TARGET])],
         eval_metric='auc', callbacks=[lgb.early_stopping(50, verbose=False)]).best_iteration_
    for tr_f, va_f in particiones(train)]))
print(f'árboles del modelo final (promedio de folds): {n_arboles}')

modelo_final = lgb.LGBMClassifier(n_estimators=n_arboles, verbose=-1, random_state=SEMILLA,
                                  monotone_constraints=restricciones(FEATS), **MEJOR)
modelo_final.fit(X(train, FEATS), train[TARGET])

p_test = calibrador.predict(modelo_final.predict_proba(X(test, FEATS))[:, 1])
pred = pd.DataFrame({'id_solicitud': test.id_solicitud, 'prob_default': p_test})
pred.to_csv('../predictions.csv', index=False)

print(f'\npredictions.csv escrito: {pred.shape[0]:,} filas')
print(f'probabilidad media predicha en test: {p_test.mean():.2%}')
print(f'rango: {p_test.min():.2%} → {p_test.max():.2%}')
assert len(pred) == 12_000 and pred.id_solicitud.is_unique and pred.prob_default.between(0, 1).all()
print('validación de formato: OK')
print(pred.head().to_string(index=False))

In [ ]:
u_test = (0.005*test.plazo_meses)/(0.005*test.plazo_meses + 0.55)
aprob = p_test < u_test

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Distribución de la probabilidad predicha en test', 'Tasa de aprobación por plazo'))
fig.add_histogram(x=p_test, nbinsx=60, marker_color=BICE_AZUL, showlegend=False, row=1, col=1)
z = pd.DataFrame({'plazo': test.plazo_meses, 'ok': aprob}).groupby('plazo')['ok'].mean()
fig.add_bar(x=z.index.astype(str), y=z.values, marker_color=BICE_ACENTO, showlegend=False,
            text=[f'{v:.0%}' for v in z.values], textposition='outside', row=1, col=2)
fig.update_xaxes(title_text='probabilidad de default', tickformat='.0%', row=1, col=1)
fig.update_xaxes(title_text='plazo (meses)', row=1, col=2)
fig.update_yaxes(title_text='% aprobado', tickformat='.0%', row=1, col=2)
fig.update_layout(title='Salida del modelo sobre las 12.000 solicitudes de test', height=430)
fig.show()

print(f'Tasa de aprobación global bajo la política recomendada: {aprob.mean():.1%}')

## Resumen del Paso 2

| Decisión | Justificación | Evidencia |
|---|---|---|
| Sin rebalanceo de clases | La política usa probabilidades absolutas; rebalancear las distorsiona | §1 |
| Baseline logística con WOE | Piso comparable y trazable; estándar de la banca | §3 |
| LightGBM como modelo principal | Captura no linealidades e interacciones; soporta monotonía; rápido para 200 ajustes | §4.1 |
| Restricciones de monotonía en 7 variables | Robustez bajo deriva, defendibilidad y auditabilidad — **sin costo de métrica** | §4.2, §4.4, §5.3 |
| Hiperparámetros elegidos sobre folds temporales | Un tuning con folds aleatorios contamina la elección | §4.3 |
| Excluir `tasa_interes_anual` | Cuesta poco AUC y compra independencia del motor de pricing | §4.4 |
| Calibración isotónica sobre predicciones out-of-fold | Usa toda la data sin calibrar contra datos vistos | §6.1 |
| **No** ajustar por la deriva; declararla | El ajuste no es validable con los datos disponibles | §6.2 |
| Umbral de aprobación por plazo | El monto se cancela en la ecuación de valor esperado | §7 |

**Performance declarada:** el AUC out-of-time promedio de los folds, con su dispersión, es la
estimación honesta. La dispersión entre folds es mayor que la diferencia entre configuraciones
de hiperparámetros: la incertidumbre relevante es temporal.

**Próximo paso:** el informe ejecutivo (`INFORME.md`), con la política recomendada, la ganancia
estimada y las limitaciones declaradas.